<a href="https://colab.research.google.com/github/kiryu-arai/kaggle_compedition_monster/blob/suzuki/v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# カリフォルニア住宅価格予測 - 発展的な特徴量エンジニアリングと交差検証パイプライン

このノートブックでは、以下の5つのアプローチを統合した高度なパイプラインを実行します。
1. **PCA（主成分分析）** によるサイズ関連変数の次元縮約
2. **主要都市（SF/LA）からの距離** の算出による地理情報の集約
3. **K-Meansクラスタリング** による地域ブロックの自動グループ化
4. **5-Fold 交差検証（Cross Validation）** による頑健な評価とアンサンブル予測
5. **目的変数（Price）の対数変換（log1p/expm1）** による予測精度の安定化

In [4]:
# kaggle APIのインストール
!pip install kaggle

In [5]:
# driveのマウント
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# パイプライン構成
import os
import json

# カレントディレクトリ、または適切なパスから kaggle.json を読み込んでください
with open("kaggle.json", 'r') as f:
    json_data = json.load(f)
os.environ['KAGGLE_USERNAME'] = json_data['username']
os.environ['KAGGLE_KEY'] = json_data['key']

In [7]:
# データのダウンロードと解凍
!kaggle competitions download -c ambl-california-housing
!unzip -o /content/ambl-california-housing.zip

100% 985k/985k [00:00<00:00, 34.5MB/s]

Archive:  /content/ambl-california-housing.zip
  inflating: sample.csv              
  inflating: test.csv                
  inflating: train.csv               


In [8]:
# データの移動
import os
import shutil

destination_folder = '/content/drive/MyDrive/kaggle_data'

if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)
    print(f"フォルダ '{destination_folder}' を作成しました。")
else:
    print(f"フォルダ '{destination_folder}' は既に存在します。")

files_to_move = ['sample.csv', 'test.csv', 'train.csv']

for file_name in files_to_move:
    source_path = os.path.join('/content/', file_name)
    destination_path = os.path.join(destination_folder, file_name)
    if os.path.exists(source_path):
        shutil.move(source_path, destination_path)
        print(f"'{file_name}' を '{destination_folder}' に移動しました。")
    else:
        print(f"'{file_name}' は存在しませんでした。")

フォルダ '/content/drive/MyDrive/kaggle_data' は既に存在します。
'sample.csv' を '/content/drive/MyDrive/kaggle_data' に移動しました。
'test.csv' を '/content/drive/MyDrive/kaggle_data' に移動しました。
'train.csv' を '/content/drive/MyDrive/kaggle_data' に移動しました。


### モジュールの準備

In [9]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb

%matplotlib inline

### データセットの読み込み

In [10]:
train = pd.read_csv('/content/drive/MyDrive/kaggle_data/train.csv')
test = pd.read_csv('/content/drive/MyDrive/kaggle_data/test.csv')
sample = pd.read_csv('/content/drive/MyDrive/kaggle_data/sample.csv')

### 特徴量エンジニアリング（FE）の定義と実行

In [11]:
def advanced_feature_engineering(train_df, test_df):
    # 訓練データとテストデータを結合して一括で処理
    df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

    # 既存のベース特徴量
    df['Household'] = df['Population'] / df['AveOccup']
    df['AllRooms'] = df['Household'] * df['AveRooms']
    df['AllBedrms'] = df['Household'] * df['AveBedrms']
    df['RoomsPerBedroom'] = df['AveRooms'] / df['AveBedrms']
    df['BedroomRatio'] = df['AveBedrms'] / df['AveRooms']

    # 【点2】緯度と経度を集約 (主要都市の中心部からのユークリッド距離)
    sf_coord = (37.7749, -122.4194)
    la_coord = (34.0522, -118.2437)
    df['dist_to_SF'] = np.sqrt((df['Latitude'] - sf_coord[0])**2 + (df['Longitude'] - sf_coord[1])**2)
    df['dist_to_LA'] = np.sqrt((df['Latitude'] - la_coord[0])**2 + (df['Longitude'] - la_coord[1])**2)

    # 【点3】K-Meansによる地理的クラスタリング (10個のエリアに分割)
    kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
    df['geo_cluster'] = kmeans.fit_predict(df[['Latitude', 'Longitude']])

    # 【点1】PCA (主成分分析) による次元縮約
    pca_cols = ['AveRooms', 'AveBedrms', 'Population', 'Household', 'AllRooms', 'AllBedrms']
    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(df[pca_cols])

    pca = PCA(n_components=2, random_state=42)
    pca_results = pca.fit_transform(scaled_features)
    df['pca_dim1'] = pca_results[:, 0]
    df['pca_dim2'] = pca_results[:, 1]

    # 多項式・交差特徴量の追加
    df['MedInc_sq'] = df['MedInc']**2
    df['HouseAge_sq'] = df['HouseAge']**2
    df['Lat_sq'] = df['Latitude']**2
    df['Lat_Lon_interaction'] = df['Latitude'] * df['Longitude']

    # 処理後に再び訓練データとテストデータに再分割
    train_fe = df[df['Price'].notnull()].copy()
    test_fe = df[df['Price'].isnull()].copy().drop(['Price'], axis=1)

    return train_fe, test_fe

print("高度な特徴量エンジニアリングを実行中...")
train_fe, test_fe = advanced_feature_engineering(train, test)
print("特徴量エンジニアリングが完了しました。")

高度な特徴量エンジニアリングを実行中...
特徴量エンジニアリングが完了しました。


### 5-Fold 交差検証と対数変換を用いたモデルの学習・評価

In [12]:
# 説明変数と目的変数の分離
features = [c for c in train_fe.columns if c not in ['Price', 'id']]
X = train_fe[features]
y = train_fe['Price']
X_test = test_fe[features]

# 【点4】交差検証 (5-Fold CV) の準備
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(train_fe))
test_preds = np.zeros(len(test_fe))

# LightGBMのハイパーパラメータ設定
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'random_state': 42,
    'n_estimators': 2000,
    'learning_rate': 0.02,
    'max_depth': 6,
    'num_leaves': 31,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'importance_type': 'gain',
    'verbosity': -1
}

print("5-Fold 交差検証による学習を開始します...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    # 【点5】目的変数（Price）の対数変換
    y_train_log = np.log1p(y_train)
    y_val_log = np.log1p(y_val)

    model = lgb.LGBMRegressor(**lgb_params)

    # 学習を実行 (early_stoppingで過学習を防止)
    model.fit(
        X_train, y_train_log,
        eval_set=[(X_val, y_val_log)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    # 検証データの予測と逆変換 (np.expm1)
    val_pred_log = model.predict(X_val)
    val_pred = np.expm1(val_pred_log)
    val_pred = np.clip(val_pred, 0, 5.00001)  # ハズレ値のクリッピング
    oof_preds[val_idx] = val_pred

    # テストデータの予測（各Foldのモデルによる予測の平均をとるアンサンブル）
    test_pred_log = model.predict(X_test)
    test_preds += np.expm1(test_pred_log) / kf.n_splits

    fold_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    print(f"Fold {fold+1} 検証データ RMSE: {fold_rmse:.5f}")

# 総合CVスコアの評価
cv_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f"\n★ 総合交差検証（CV）の平均 RMSE: {cv_rmse:.5f}")

5-Fold 交差検証による学習を開始します...
Fold 1 検証データ RMSE: 0.46181
Fold 2 検証データ RMSE: 0.44294
Fold 3 検証データ RMSE: 0.44902
Fold 4 検証データ RMSE: 0.43052
Fold 5 検証データ RMSE: 0.43614

★ 総合交差検証（CV）の平均 RMSE: 0.44422


### 提出用ファイルの作成と保存

In [13]:
sample_cv = sample.copy()
sample_cv['Price'] = test_preds
output_path = '/content/drive/MyDrive/kaggle_data/submit_cv_advanced.csv'
sample_cv.to_csv(output_path, index=None)

print(f"提出用ファイル '{output_path}' を正常に作成しました。")
display(sample_cv.head())

提出用ファイル '/content/drive/MyDrive/kaggle_data/submit_cv_advanced.csv' を正常に作成しました。


,id,Price
0,0,2.779885
1,1,1.787247
2,2,0.952279
3,3,3.886559
4,4,3.955114


### Kaggleへの直接投稿

In [14]:
# 作成したファイルをKaggleに直接投稿
!kaggle competitions submit -c ambl-california-housing -f /content/drive/MyDrive/kaggle_data/submit_cv_advanced.csv -m "Advanced FE and 5-Fold CV Submission via Colab"

100% 94.2k/94.2k [00:00<00:00, 467kB/s]
Successfully submitted to AMBL初心者向けコンペティション_California Housing